# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 1. Carga y validación de los datos

En esta primera etapa se cargan y exploran los datasets principales de RappiPlus para evaluar su estructura, tipos de datos, valores faltantes y posibles inconsistencias antes de realizar las transformaciones y análisis posteriores.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [45]:
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [46]:
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [47]:
# explorar datasets catalog
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [48]:
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [49]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [50]:
# explorar datasets marketing
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [51]:
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [52]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

### Conversión de fechas

Las variables de fecha se convierten al formato datetime para facilitar el análisis temporal y garantizar un tratamiento consistente de las fechas inválidas.

In [53]:
#Validar y convertir fechas
def convertir_fechas(df, columnas):
    for col in columnas:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

orders = convertir_fechas(orders, ['fecha_hora_pedido'])
marketing = convertir_fechas(marketing, ['fecha'])

### Tratamiento de valores negativos

Durante la exploración inicial se identificaron valores negativos en `cantidad` y `monto_total`. Antes de transformarlos, se conservan los registros originales para evaluar si corresponden a devoluciones, cancelaciones o errores de captura.

Para evitar alterar la semántica de los datos, los registros negativos se identifican mediante una bandera y se revisan antes de decidir su tratamiento.

In [ ]:
orders['es_valor_negativo'] = (
    (orders['cantidad'] < 0) |
    (orders['monto_total'] < 0)
)

orders[orders['es_valor_negativo']].head()

<div class="alert alert-block alert-warning">
<b>Comentario del Revisor - 1ª Iteración</b> <a class="tocSkip"></a><br>

<b>Atención</b> ⚠️ - Al convertir fechas y eliminar duplicados validaste aspectos clave: usaste <code>pd.to_datetime</code>, comprobaste consistencia de montos y eliminaste duplicados. Sin embargo, la decisión de corregir valores negativos con <code>abs()</code> y después convertir ceros a <code>NaN</code> y eliminar filas puede sesgar el análisis (por ejemplo, elimina devoluciones o pedidos nulos que requieren tratamiento distinto). 


Propuesta:  <ul><li>Inspecciona las filas con valores negativos antes de transformarlas: <code>orders.loc[orders['cantidad'] &lt; 0 | orders['monto_total'] &lt; 0]</code>.</li><li>Si representan devoluciones, crea una columna <code>tipo_transaccion</code> o una bandera <code>es_retorno</code> en vez de tomar el valor absoluto.</li><li>Antes de convertir ceros a <code>NaN</code> y borrarlas, valida caso por caso: <code>orders[orders['monto_total']==0].head()</code>. Si son válidas, imputarlas o tratarlas como categoría explícita.</li></ul>Por qué importa: conservar la semántica de retornos y ceros evita inflar métricas como revenue o ticket promedio. Buen trabajo identificando inconsistencias; con esos pasos dejarás la limpieza lista para producción.

</div>

In [55]:
#Ceros aunque no hay ceros
def corregir_ceros(df, columnas):
    for col in columnas:
        df.loc[df[col] == 0, col] = np.nan
    return df
orders = corregir_ceros(orders, ['monto_total', 'cantidad', 'precio_unitario'])

In [56]:
#Verificar consistencia de montos
orders["monto_consistente"] = np.isclose(
    orders['monto_total'],
    (orders['precio_unitario'] * orders['cantidad']) - orders['monto_descuento'], atol=0.01)    
print(orders['monto_consistente'].value_counts())

True     25050
False       50
Name: monto_consistente, dtype: int64


### Eliminación de registros duplicados

Se identifican y eliminan registros completamente duplicados para evitar que una misma transacción contribuya más de una vez al análisis.

In [ ]:
duplicados_antes = orders.duplicated().sum()
print(f'Duplicados encontrados: {duplicados_antes}')

orders = orders.drop_duplicates()

print(f'Duplicados después de la limpieza: {orders.duplicated().sum()}')

Cantidad de duplicados:  0


In [59]:
#Revisar variables categóricas
def limpiar_texto(df, columnas):
    for col in columnas:
        df[col] = df[col].str.strip().str.lower()
    return df

categoricas_marketing=['pais', 'canal']
orders=limpiar_texto(orders, ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto'])
catalog=limpiar_texto(catalog, ['categoria_producto'])
marketing=limpiar_texto(marketing, ['pais', 'canal'] )

In [ ]:
# Imputar valores faltantes en variables categóricas
def imputar_desconocido(df):
    columnas=['pais', 'dispositivo', 'fuente_referencia']
    for columna in columnas:
        df[columna] = df[columna].fillna('Desconocido')
    return df
orders = imputar_desconocido(orders)

### Resultado de la revisión

Los valores faltantes representan menos del 3% de los registros. Debido a su baja proporción, se opta por conservar los registros siempre que sea posible y aplicar un tratamiento específico según el tipo de variable, evitando eliminar filas de manera indiscriminada.

In [ ]:
filas_antes = len(orders)
orders.dropna(inplace=True)
filas_despues = len(orders)

print(f"Se eliminaron {filas_antes - filas_despues} filas.")

In [62]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 2: Análisis de rentabilidad 

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [ ]:
#Rentabilidad del negocio
orders_union = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
orders_union['costo_unitario'].isna().sum()

0

In [64]:
#Costo total columna
orders_union['costo_total'] = (
    orders_union['cantidad'] * orders_union['costo_unitario'])

#Ingreso total (revenue)
revenue = orders_union['monto_total'].sum()
print(f' Ingreso Total: ${revenue: ,.2f}')

#Costo total
costo_total = orders_union['costo_total'].sum()
print(f'\n Costo Total: ${costo_total: ,.2f}')

#Inversión marketing
marketing_inversion = marketing['gasto'].sum()
print(f'\n Inversión Marketing: ${marketing_inversion: ,.2f}')

#Profit
profit = revenue - costo_total - marketing_inversion
print(f'\n Profit: ${profit: ,.2f}')

 Ingreso Total: $ 51,955,866.24

 Costo Total: $ 43,124,119.61

 Inversión Marketing: $ 2,871,843.53

 Profit: $ 5,959,903.10


In [65]:
# Comportamiento de ventas

#Ticket promedio por orden
ticket_promedio = orders_union['monto_total'].mean()
print(f' Ticket Promedio: {ticket_promedio: .2f}')

#Cantidad promedio de productos por orden
cantidad_promedio = orders_union['cantidad'].mean()
print(f'\n Cantidad Promedio De Productos Por Orden: {cantidad_promedio: .2f}')

 Ticket Promedio:  2084.91

 Cantidad Promedio De Productos Por Orden:  7.12


In [66]:
#Producto mas vendido
producto_mas_vendido = (
    orders_union.groupby('nombre_producto')['cantidad']
    .sum().sort_values(ascending=False).head(1)
)
print(f' Producto Mas Vendido: \n  {producto_mas_vendido}')

 Producto Mas Vendido: 
  nombre_producto
Laptop-Gaming-16GB    144198.0
Name: cantidad, dtype: float64


In [67]:
#Gasto marketing por canal
gasto_marketing_canal = (
    marketing.groupby('canal')['gasto']
    .sum().sort_values(ascending=False)
)
print(gasto_marketing_canal)

canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


---

## 3. Funnel de conversión

### Objetivo

Analizar el comportamiento de los usuarios a lo largo del proceso de conversión para identificar las etapas con mayor pérdida de usuarios y calcular la conversión final.

### Construcción del funnel

Se utilizan los eventos registrados en la tabla `events` para:

- calcular usuarios únicos en cada etapa;
- ordenar los eventos de acuerdo con el flujo de conversión;
- construir el funnel de usuarios.

### Análisis de conversión

Se calcula la tasa de conversión entre etapas consecutivas y se identifica el punto del proceso donde se presenta la mayor pérdida de usuarios.

El análisis se realiza mediante consultas SQL sobre la base de datos.

### Conexión a la base de datos

Las consultas SQL se ejecutan mediante SQLAlchemy sobre una base de datos PostgreSQL. Las credenciales de acceso se gestionan mediante variables de entorno y no se almacenan en el notebook.

In [ ]:
import os
from sqlalchemy import create_engine

connection_string = os.getenv("RAPPIPLUS_DB_URL")

engine = create_engine(
    connection_string,
    connect_args={"sslmode": "require"}
)

In [69]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [70]:
# Explorar tabla events
# =========================
query_distinct = '''
SELECT DISTINCT nombre_evento
FROM events;
'''
distinct = pd.read_sql(query_distinct, con=engine)
distinct

,nombre_evento
0,add_payment_info
1,first_visit
2,begin_checkout
3,add_to_cart
4,select_item
5,purchase


In [71]:
# PARTE 1: Totales del funnel
#Cuantos usuarios ùnicos realizaron cada evento
# ======================


query_totales = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY CASE
    WHEN nombre_evento = 'first_visit' THEN 1
    WHEN nombre_evento = 'select_item' THEN 2
    WHEN nombre_evento = 'add_to_cart' THEN 3
    WHEN nombre_evento = 'begin_checkout' THEN 4
    WHEN nombre_evento = 'add_payment_info' THEN 5
    WHEN nombre_evento = 'purchase' THEN 6
END;
'''

totales = pd.read_sql(query_totales, con=engine)
totales

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [87]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'select_item' THEN 2
            WHEN nombre_evento = 'add_to_cart' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END AS orden
    FROM events
    GROUP BY nombre_evento
)

SELECT
    nombre_evento,
    usuarios,
    LAG(usuarios) OVER (ORDER BY orden) AS usuarios_etapa_anterior,
    ROUND(
        usuarios * 100.0 /
        LAG(usuarios) OVER (ORDER BY orden),
        2
    ) AS tasa_conversion

FROM funnel
ORDER BY orden;

'''

conversion_funnel = pd.read_sql(query_conversion, con=engine)
conversion_funnel

,nombre_evento,usuarios,usuarios_etapa_anterior,tasa_conversion
0,first_visit,7796,NaN,NaN
1,select_item,7582,7796.0,97.26
2,add_to_cart,7634,7582.0,100.69
3,begin_checkout,7208,7634.0,94.42
4,add_payment_info,6250,7208.0,86.71
5,purchase,6240,6250.0,99.84


In [88]:
#Tasa de conversion final
query_conversion_final = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios
    FROM events
    GROUP BY nombre_evento
)

SELECT
    ROUND(
        purchase.usuarios * 100.0 / first_visit.usuarios,
        2
    ) AS conversion_final
FROM funnel AS purchase
JOIN funnel AS first_visit
ON 1 = 1
WHERE purchase.nombre_evento = 'purchase'
AND first_visit.nombre_evento = 'first_visit';
'''

conversion_final = pd.read_sql(query_conversion_final, con=engine)
conversion_final

,conversion_final
0,80.04


## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [74]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [75]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [76]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        u.id_usuario,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE))::DATE AS cohorte,
        ua.dias_despues_registro,
        ua.activo,
        FLOOR(ua.dias_despues_registro / 7) AS semana
    FROM users u
    LEFT JOIN user_activity ua
        ON u.id_usuario = ua.id_usuario
),
retencion AS (
    SELECT
        cohorte,
        COUNT(DISTINCT id_usuario) AS clientes,
        COUNT(DISTINCT CASE WHEN
            semana = 0
            AND activo = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN
            semana = 1
            AND activo = 1 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN
            semana = 2
            AND activo = 1 THEN id_usuario END) AS retenido_w3
    FROM cohortes
    GROUP BY cohorte
)

SELECT
    cohorte,
    clientes,
    retenido_w1,
    ROUND(retenido_w1 * 100 / clientes, 2) AS semana_1,
    retenido_w2,
    ROUND(retenido_w2 * 100 / clientes, 2) AS semana_2,
    retenido_w3,
    ROUND(retenido_w3 * 100 / clientes, 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes,retenido_w1,semana_1,retenido_w2,semana_2,retenido_w3,semana_3
0,2025-01-01,1627,0,0.0,697,42.0,668,41.0
1,2025-02-01,1444,0,0.0,611,42.0,609,42.0
2,2025-03-01,1636,0,0.0,677,41.0,705,43.0
3,2025-04-01,1606,0,0.0,680,42.0,697,43.0
4,2025-05-01,1687,0,0.0,695,41.0,676,40.0


---

## 5. Test estadístico: impacto de cambios en el checkout

### Objetivo

Evaluar si una modificación en la interfaz del checkout genera un cambio estadísticamente significativo en la tasa de conversión de compra.

### Metodología

Se analiza un experimento A/B en el que los usuarios fueron asignados a una variante de control o tratamiento.

La métrica principal es la **tasa de conversión**, definida como la proporción de usuarios que completaron una compra.

El análisis comprende:

1. Exploración de las variantes y la tasa de conversión.
2. Formulación de las hipótesis estadísticas.
3. Aplicación de una prueba estadística para comparar las proporciones.
4. Interpretación del valor p y de la evidencia disponible para determinar si existe una diferencia estadísticamente significativa entre las variantes.

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [ ]:
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [78]:
experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [103]:
# Número de usuarios convertidos por página
conversiones = experiment.groupby('variante')['convirtio'].sum()
conversiones
# Total de usuarios por página
totales=experiment.groupby('variante')['convirtio'].count()
totales
print("Usuarios convertidos por variante:\n", conversiones)
print("\nTotal de usuarios por variante:\n", totales)

Usuarios convertidos por variante:
 variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

Total de usuarios por variante:
 variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64


In [106]:
exitos=[conversiones['control'], conversiones['tratamiento']]
print(exitos)
observaciones=[totales['control'], totales['tratamiento']]
print(observaciones)

[779, 820]
[4965, 5035]


In [108]:
#Hipotesis Nula H0 La tasa de conversion es igual entre las variantes
#Hipótesis Alternativa H1 La tasa de conversión es diferente entre las variantes

from statsmodels.stats.proportion import proportions_ztest
z_stat, p_value = proportions_ztest(exitos, observaciones)
# Visualizar resultados
print(f"Estadístico : {z_stat}")
print(f"Valor p: {p_value}")

Estadístico : -0.8132782986429474
Valor p: 0.41605851639119995


In [109]:
tasa_control= exitos[0] / observaciones [0]
print(f" tasa de conversión control: {tasa_control: .2%}")
tasa_tratamiento= exitos[1] / observaciones [1]
print(f" tasa de conversión tratamiento: {tasa_tratamiento: .2%}")

 tasa de conversión control:  15.69%
 tasa de conversión tratamiento:  16.29%


In [112]:
alpha = 0.05
if p_value < alpha:
    print('Se rechaza la hipótesis nula. Existe una diferencia estadísticamente significativa')
else:
    print('No se rechaza la hipótesis nula. No existe eviedencia suficiente para afirmar que hay diferencia')

No se rechaza la hipótesis nula. No existe eviedencia suficiente para afirmar que hay diferencia


---

---

## Dashboard en Power BI

Se desarrolló un dashboard ejecutivo para analizar el desempeño comercial de RappiPlus.

El dashboard permite analizar:

- Revenue y profit.
- Evolución mensual de las ventas.
- Desempeño por categoría y producto.
- Gasto de marketing.
- Detalle de las órdenes.
- Rentabilidad por producto.

El dashboard incluye una vista ejecutiva y una página de detalle con navegación mediante drill-through.

# (https://drive.google.com/drive/folders/11HSU3mSOS4C78oAscdBCsYw2iOjz90II?usp=drive_link)
# link de one drive / google drive